In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os

# Surprise library
import surprise
from surprise import Dataset, Reader, SVD, SVDpp, NMF, KNNBaseline
from surprise import accuracy
from surprise.prediction_algorithms.predictions import Prediction
from surprise.model_selection import GridSearchCV, KFold
from tqdm import tqdm

## Data Prep

- add ratings to probe set from training set
- drop probe entries from training set
- store train and probe set into parque

In [4]:
PROJECT_ROOT = Path.cwd().parent

# add ratings to probe set from training set
DATA_DIR = PROJECT_ROOT / "data"
RATINGS_PATH = DATA_DIR / "ratings.parquet"
PROBE_PATH = DATA_DIR / "probe.parquet"
ratings = pd.read_parquet(RATINGS_PATH)
probe = pd.read_parquet(PROBE_PATH)
ratings['_hash'] = ratings['CustomerID'].astype(np.int64) * 10**6 + ratings['MovieID'].astype(np.int64)
probe['_hash'] = probe['CustomerID'].astype(np.int64) * 10**6 + probe['MovieID'].astype(np.int64)

probe_hash_set = set(probe['_hash'].values)

# split train and probe set
mask_probe = np.isin(ratings['_hash'].values, list(probe_hash_set))
probe_with_ratings = ratings[mask_probe].copy()
train = ratings[~mask_probe].copy()

train.drop(columns=['_hash'], inplace=True)
probe_with_ratings.drop(columns=['_hash'], inplace=True)

print(f"Train size: {len(train):,}, Probe size: {len(probe_with_ratings):,}")

Train size: 99,072,112, Probe size: 1,408,395


In [5]:
# save train and probe with ratings
train_path = os.path.join(DATA_DIR, "train.parquet")
probe_path = os.path.join(DATA_DIR, "probe_with_ratings.parquet")

train.to_parquet(train_path, index=False)
probe_with_ratings.to_parquet(probe_path, index=False)

print(f"Saved train to: {train_path}")
print(f"Saved probe to: {probe_path}")

Saved train to: /Users/amily/Desktop/netflix-prize-data-mining-project/data/train.parquet
Saved probe to: /Users/amily/Desktop/netflix-prize-data-mining-project/data/probe_with_ratings.parquet


In [6]:
# select only CustomerID, MovieID, Rating
probe = probe_with_ratings[["CustomerID", "MovieID", "Rating"]]
train = train[["CustomerID", "MovieID", "Rating"]]

## Baseline and Bias model

- **Global mean**: Single constant prediction (training mean); baseline for comparison.
- **User + Movie bias**: prediction = global_mean + user_bias + movie_bias; captures per-user and per-movie offsets.

In [7]:
# Helper method
def rmse(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [8]:
# Global mean prediction
global_mean = train["Rating"].mean()

# RMSE helper already defined above
probe_global_pred = np.full(len(probe), global_mean)
rmse_global = rmse(probe["Rating"], probe_global_pred)
print(f"Global-mean baseline RMSE on probe: {rmse_global:.4f}")

# User and Movie bias model (fit on TRAIN only)
user_mean = train.groupby("CustomerID")["Rating"].mean()
movie_mean = train.groupby("MovieID")["Rating"].mean()

user_bias = user_mean - global_mean
movie_bias = movie_mean - global_mean

# Map biases to probe set; unseen users/movies get bias 0
bu = probe["CustomerID"].map(user_bias).fillna(0.0)
bi = probe["MovieID"].map(movie_bias).fillna(0.0)

probe_bias_pred = global_mean + bu + bi
rmse_bias = rmse(probe["Rating"], probe_bias_pred)
print(f"User+Movie bias model RMSE on probe: {rmse_bias:.4f}")

Global-mean baseline RMSE on probe: 1.1296
User+Movie bias model RMSE on probe: 0.9965


# Ensemble Recommendation Model
- Base: Matrix Factorization (SVD, NMF)
- Enhancement: Residual correction using Item-based Collaborative Filtering (Item-CF)

### Matrix factorization models

- **SVD**: Factorizes the rating matrix into user and item latent factors; optimized with SGD and L2 regularization. Best-performing single model here.
- **NMF**: Same low-rank idea with non-negative constraints on factors.

Training workflow:
1. Hyperparameter tuning (grid search + 3-fold CV) on a 50k-sample subset
2. refit best params on full training set
3. evaluate on probe set.

In [ ]:
# Subsample TRAIN for hyperparameter tuning (to keep runtime <~2 hours)
TUNE_SAMPLE_N = 50_000  # target number of train ratings used for CV

if len(train) > TUNE_SAMPLE_N:
    train_tune = train.sample(n=TUNE_SAMPLE_N, random_state=42)
else:
    train_tune = train.copy()

print(f"Tuning on {len(train_tune):,} ratings out of {len(train):,} train ratings.")

# Build Surprise datasets
reader = Reader(rating_scale=(float(train["Rating"].min()), float(train["Rating"].max())))

data_tune = Dataset.load_from_df(
    train_tune[["CustomerID", "MovieID", "Rating"]], reader
)


data_full = Dataset.load_from_df(
    train[["CustomerID", "MovieID", "Rating"]], reader
)
full_trainset = data_full.build_full_trainset()

# 3-fold CV on tuning subset
cv = KFold(n_splits=3, random_state=42, shuffle=True)


def tune_and_evaluate(algo_class, param_grid, algo_name):
    """3-fold CV on a tuning subset, then refit best model on full TRAIN and evaluate on PROBE."""
    grid = GridSearchCV(
        algo_class,
        param_grid,
        measures=["rmse"],
        cv=cv,
        n_jobs=-1,
        joblib_verbose=2,  # show progress
    )
    grid.fit(data_tune)

    best_params = grid.best_params["rmse"]
    best_cv_rmse = grid.best_score["rmse"]
    print(f"{algo_name} best 3-fold CV RMSE (tune subset): {best_cv_rmse:.4f}")
    print(f"{algo_name} best params: {best_params}")

    # Refit best model on full train
    best_algo = algo_class(**best_params)
    print("fit")
    best_algo.fit(full_trainset)
    return best_algo, best_params, best_cv_rmse

Tuning on 50,000 ratings out of 99,072,112 train ratings.


In [ ]:
svd_param_grid = {
    "n_factors": [20, 50],
    "lr_all": [0.005],
    "reg_all": [0.02, 0.05],
    "n_epochs": [5],
}
best_svd, best_svd_params, best_svd_cv_rmse = tune_and_evaluate(
    SVD, svd_param_grid, "SVD"
)
svd_result = pd.DataFrame([{
    "Model": "SVD",
    "Best_Params": best_svd_params,
    "CV_RMSE": best_svd_cv_rmse
}])
svd_result

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "03_recommendation"
os.makedirs(OUTPUT_DIR, exist_ok=True)
svd_path = OUTPUT_DIR / "best_svd.pkl"

# Save the model
with open(svd_path, "wb") as f:
    pickle.dump(best_svd, f)

print(f"Saved best SVD model to: {svd_path}")

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   8 | elapsed:    2.2s remaining:    3.6s
[Parallel(n_jobs=-1)]: Done   8 out of   8 | elapsed:    2.4s finished


SVD best 3-fold CV RMSE (tune subset): 1.0596
SVD best params: {'n_factors': 20, 'lr_all': 0.005, 'reg_all': 0.02, 'n_epochs': 5}
fit


In [8]:
svd_result

,Model,Best_Params,CV_RMSE
0,SVD,"{'n_factors': 20, 'lr_all': 0.005, 'reg_all': ...",1.059587


In [ ]:
nmf_param_grid = {
    "n_factors": [20, 50],
    "reg_pu": [0.02],
    "reg_qi": [0.02],
    "n_epochs": [5],
}

best_nmf, best_nmf_params, best_nmf_cv_rmse = tune_and_evaluate(
    NMF, nmf_param_grid, "NMF"
)
nmf_result = pd.DataFrame([{
    "Model": "NMF",
    "Best_Params": best_nmf_params,
    "CV_RMSE": best_nmf_cv_rmse
}])

nmf_path = OUTPUT_DIR / "best_nmf.pkl"

# Save the model
with open(nmf_path, "wb") as f:
    pickle.dump(best_nmf, f)

print(f"Saved best NMF model to: {nmf_path}")

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of   4 | elapsed:    2.3s finished


NMF best 3-fold CV RMSE (tune subset): 1.1677
NMF best params: {'n_factors': 20, 'reg_pu': 0.02, 'reg_qi': 0.02, 'n_epochs': 5}
fit


In [10]:
nmf_result

,Model,Best_Params,CV_RMSE
0,NMF,"{'n_factors': 20, 'reg_pu': 0.02, 'reg_qi': 0....",1.167726


In [8]:
testset = list(zip(
        probe["CustomerID"].values,
        probe["MovieID"].values,
        probe["Rating"].values.astype(float)
    ))
predictions = best_nmf.test(testset)
rmse = accuracy.rmse(predictions, verbose=True)

RMSE: 1.4856


In [12]:
testset = list(zip(
        probe["CustomerID"].values,
        probe["MovieID"].values,
        probe["Rating"].values.astype(float)
    ))
predictions = best_svd.test(testset)
rmse = accuracy.rmse(predictions, verbose=True)

RMSE: 0.9632


### Item-based CF correction on best SVD (1000 popular movies)

Hybrid that uses item-based KNN to correct SVD on popular movies:
- **Residuals**: For each rating in `train_knn` (top popular movies), compute residual = actual − SVD prediction.
- **KNN on residuals**: Fit item-based KNN (e.g. KNNBaseline) to predict these residuals; only items in the popular set.
- **Final prediction**: SVD prediction + α × KNN residual when the movie is popular (α tunable; 0 = SVD-only). Predictions are clipped to [1, 5].

In [ ]:
# select top 1000 popular movies
movie_counts = train.groupby("MovieID").size()
popular_movies = (
    train.groupby("MovieID")
    .size()
    .sort_values(ascending=False)
    .head(1000)
    .index
)
train_knn = train[train["MovieID"].isin(popular_movies)]

In [29]:
# Compute SVD residuals on train_knn
RESIDUAL_SAMPLE = None
train_for_residual = (
    train_knn.sample(n=min(RESIDUAL_SAMPLE, len(train_knn)), random_state=42)
    if RESIDUAL_SAMPLE and len(train_knn) > RESIDUAL_SAMPLE
    else train_knn
)

CHUNK_SIZE = 100_000
residuals_list = []

n_chunks = len(train_for_residual) // CHUNK_SIZE + 1

for start in tqdm(range(0, len(train_for_residual), CHUNK_SIZE), total=n_chunks, desc="Computing residuals"):
    chunk = train_for_residual.iloc[start : start + CHUNK_SIZE]

    svd_preds = [
        best_svd.predict(uid, iid).est
        for (uid, iid) in zip(chunk["CustomerID"], chunk["MovieID"])
    ]

    res = chunk["Rating"].values.astype(float) - np.array(svd_preds)

    residuals_list.append(
        pd.DataFrame({
            "CustomerID": chunk["CustomerID"].values,
            "MovieID": chunk["MovieID"].values,
            "Residual": res,
        })
    )

residuals_df = pd.concat(residuals_list, ignore_index=True)

print(f"Computed residuals for {len(residuals_df):,} pairs")
print(f"Residual stats: mean={residuals_df['Residual'].mean():.4f}, std={residuals_df['Residual'].std():.4f}")

Computing residuals: 100%|██████████| 621/621 [07:21<00:00,  1.41it/s]


Computed residuals for 62,014,880 pairs
Residual stats: mean=-0.0984, std=0.8707


In [ ]:
# implement item based CF from scratch for faster computation
residuals_df["user_idx"] = residuals_df["CustomerID"].astype("category").cat.codes
residuals_df["movie_idx"] = residuals_df["MovieID"].astype("category").cat.codes

user_codes = residuals_df["user_idx"].values
movie_codes = residuals_df["movie_idx"].values
ratings = residuals_df["Residual"].values

R = csr_matrix((ratings, (user_codes, movie_codes)))


# Item-item similarity
R_item = R.T
similarity = cosine_similarity(R_item, dense_output=False)

# Keep top-k neighbors
k = 40
top_k = np.argsort(-similarity.toarray(), axis=1)[:, :k]


# Map probe IDs
user_map = dict(zip(residuals_df["CustomerID"].astype("category").cat.categories, 
                    range(len(residuals_df["CustomerID"].astype("category").cat.categories))))
movie_map = dict(zip(residuals_df["MovieID"].astype("category").cat.categories, 
                     range(len(residuals_df["MovieID"].astype("category").cat.categories))))

probe_user_idx = probe["CustomerID"].map(user_map)
probe_movie_idx = probe["MovieID"].map(movie_map)

# Vectorized residual prediction with progress bar
def knn_residual_vectorized(u_idx, m_idx):
    pred = []
    for ui, mi in tqdm(zip(u_idx, m_idx), total=len(u_idx), desc="KNN residual prediction"):
        if pd.isna(ui) or pd.isna(mi):
            pred.append(0.0)
            continue
        neighbors = top_k[int(mi)]
        sims = similarity[int(mi), neighbors].toarray().flatten()
        ratings = R[int(ui), neighbors].toarray().flatten()
        mask = ratings != 0
        if mask.sum() == 0:
            pred.append(0.0)
        else:
            pred.append(np.dot(sims[mask], ratings[mask]) / np.sum(np.abs(sims[mask])))
    return np.array(pred)

knn_preds = knn_residual_vectorized(probe_user_idx.values, probe_movie_idx.values)

# SVD predictions with progress bar
svd_preds = np.array([
    best_svd.predict(uid, iid).est
    for uid, iid in tqdm(zip(probe["CustomerID"], probe["MovieID"]), total=len(probe), desc="SVD prediction")
])

# Final ensemble
final_preds = svd_preds + knn_preds
final_preds = svd_preds + 0.3 * knn_preds
probe["Predicted"] = final_preds


SVD prediction: 100%|██████████| 1408395/1408395 [00:09<00:00, 144901.46it/s]


In [ ]:
# Compute RMSE on probe set
testset = list(zip(probe["CustomerID"], probe["MovieID"], probe["Rating"].astype(float)))
predictions = [
    Prediction(uid, iid, true_r, est, 0) 
    for (uid, iid, true_r), est in tqdm(zip(testset, final_preds), total=len(testset), desc="Probe RMSE")
]
rmse = accuracy.rmse(predictions)
print(f"Probe RMSE: {rmse:.4f}")

Probe RMSE: 100%|██████████| 1408395/1408395 [00:33<00:00, 41952.03it/s] 


RMSE: 0.9491
Probe RMSE: 0.9491


# Result Summary

In [3]:
results = {
    "Model": [
        "Global-mean baseline",
        "User+Movie bias",
        "NMF",
        "SVD",
        "SVD + Item-CF residual correction"
    ],
    "Probe_RMSE": [
        1.1296,
        0.9965,
        1.4856,
        0.9632,
        0.9491
    ]
}

rmse_df = pd.DataFrame(results)
rmse_df = rmse_df.sort_values(by="Probe_RMSE").reset_index(drop=True)

rmse_df

,Model,Probe_RMSE
0,SVD + Item-CF residual correction,0.9491
1,SVD,0.9632
2,User+Movie bias,0.9965
3,Global-mean baseline,1.1296
4,NMF,1.4856


## Description of the models

| Model | Description |
|-------|-------------|
| **Global mean** | Baseline: predicts the training-set mean rating for every user–movie pair. Used as a reference. |
| **User + Movie bias** | Additive model: prediction = global_mean + user_bias + movie_bias. Captures systematic offsets (e.g. harsh vs lenient users, popular vs niche movies). |
| **SVD** | Matrix factorization: approximates the rating matrix as the product of user and item latent factor matrices. Fit with stochastic gradient descent; learns low-rank structure and generalizes well. |
| **NMF** | Non-negative matrix factorization: same idea as SVD but with non-negative constraints on factors. Often used for interpretability. |
| **Item-based KNN correction** | Item-based collaborative filtering trained on **residuals** (actual rating − SVD prediction) for popular movies only. Final prediction = SVD prediction + α × KNN residual, clipped to [1, 5]. Used to correct SVD on high-activity items. |